In [2]:
import pandas as pd
from supervised.automl import AutoML
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
from tqdm import TqdmWarning
from sklearn.model_selection import train_test_split

# Removes annoying tqdm warnings
warnings.filterwarnings("ignore", category=TqdmWarning)

In [3]:
# Constants
RANDOM_STATE = 1082
TEST_SIZE = 0.2
MODEL_TIME_LIMIT = 60 * 60 # 1 hour

In [4]:
# Load the dataset
try:
    df = pd.read_csv("../../Datasets/csv/spoofing-merged-gps-only.csv")
except FileNotFoundError:
    print("Error: The file '../../Datasets/csv/spoofing-merged-gps-only.csv' was not found.")
    print("Please make sure the path is correct.")
    exit()

In [5]:
# Data preprocessing

# Clean up column names
df.columns = df.columns.str.replace('[\\[\\]<>]', '_', regex=True)

# Drop timestamp column, keep 'label' as target variable
drop_cols = ['timestamp']
X = df.drop(columns=drop_cols + ['label'])
y = df['label']

# Encode label
if y.dtype == object or y.dtype.name == 'category':
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    y = le.fit_transform(y)
    
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

In [6]:
# Train the AutoML models

automl = AutoML(mode="Explain", random_state=RANDOM_STATE, results_path="explainResults", model_time_limit=MODEL_TIME_LIMIT)
automl.fit(X_train, y_train)

AutoML directory: explainResults
The task is binary_classification with evaluation metric logloss
AutoML will use algorithms: ['Baseline', 'Linear', 'Decision Tree', 'Random Forest', 'Xgboost', 'Neural Network']
AutoML will ensemble available models
AutoML steps: ['simple_algorithms', 'default_algorithms', 'ensemble']
* Step simple_algorithms will try to check up to 3 models
1_Baseline logloss 0.401192 trained in 0.3 seconds
2_DecisionTree logloss 0.0 trained in 13.02 seconds
3_Linear logloss 0.001686 trained in 5.4 seconds
* Step default_algorithms will try to check up to 3 models
4_Default_Xgboost logloss 0.000818 trained in 5.89 seconds
5_Default_NeuralNetwork logloss 7.2e-05 trained in 2.44 seconds
6_Default_RandomForest logloss 0.0 trained in 2.85 seconds
* Step ensemble will try to check up to 1 model
Ensemble logloss 0.0 trained in 1.84 seconds
AutoML fit time: 40.64 seconds
AutoML best model: 2_DecisionTree


AutoML(model_time_limit=3600, random_state=1082, results_path='explainResults')

In [10]:
y_pred = automl.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 1.0000

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       625
           1       1.00      1.00      1.00       100

    accuracy                           1.00       725
   macro avg       1.00      1.00      1.00       725
weighted avg       1.00      1.00      1.00       725


Confusion Matrix:
 [[625   0]
 [  0 100]]


In [9]:
print("\nAutoML Model Leaderboard:")
print(automl.get_leaderboard())


AutoML Model Leaderboard:
                      name      model_type metric_type  metric_value  \
0               1_Baseline        Baseline     logloss  4.011921e-01   
1           2_DecisionTree   Decision Tree     logloss  1.192093e-07   
2                 3_Linear          Linear     logloss  1.685987e-03   
3        4_Default_Xgboost         Xgboost     logloss  8.176748e-04   
4  5_Default_NeuralNetwork  Neural Network     logloss  7.165203e-05   
5   6_Default_RandomForest   Random Forest     logloss  1.192093e-07   
6                 Ensemble        Ensemble     logloss  1.192093e-07   

   train_time  
0        0.99  
1       13.98  
2        6.45  
3        6.97  
4        3.27  
5        3.70  
6        1.84  
